# Prompt Ablation - contraTICO
## 2) BT QA — Spanish (es)

Generate bt answers for **es** with P1-fewshot, P2-cot, P3-concise for all 3 configs × 8 perturbations.

In [ ]:
import os
import subprocess

# ─── Paths (Kaggle) ───
PROJECT_ROOT = "/kaggle/input/askqe-dnlp"
BASELINE_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/contratico/baseline"
CONTRATICO_DIR = f"{PROJECT_ROOT}/contratico"
OUTPUT_DIR = f"/kaggle/working/prompt-ablation"
CODE_DIR = f"{PROJECT_ROOT}/results Qwen3B baseline/contratico/prompt-ablation/code"

LANG = "es"
STRATEGIES = ["P1-fewshot", "P2-cot", "P3-concise"]
CONFIGS = ["vanilla", "atomic", "semantic"]

print(f"Language: {LANG}")
print(f"Baseline: {BASELINE_DIR}")
print(f"contraTICO data: {CONTRATICO_DIR}")
print(f"Output: {OUTPUT_DIR}")

In [ ]:
for strategy in STRATEGIES:
    for config in CONFIGS:
        print(f"\n{'=' * 60}")
        print(f"BT QA: {strategy} / {config} / {LANG}")
        print(f"{'=' * 60}")
        
        result = subprocess.run(
            [
                "python", f"{CODE_DIR}/qa_ablation_contratico.py",
                "--strategy", strategy,
                "--mode", "bt",
                "--config", config,
                "--lang", LANG,
                "--baseline_dir", BASELINE_DIR,
                "--contratico_dir", CONTRATICO_DIR,
                "--output_dir", OUTPUT_DIR,
                "--max_rows", "42",
                "--seed", "42",
            ],
            capture_output=True, text=True
        )
        print(result.stdout[-2000:] if len(result.stdout) > 2000 else result.stdout)
        if result.returncode != 0:
            print(f"ERROR: {result.stderr[-500:]}")

In [ ]:
# Verify output files
PERTURBATIONS = ["alteration", "expansion_impact", "expansion_noimpact",
                 "intensifier", "omission", "spelling", "synonym", "word_order"]

for strategy in STRATEGIES:
    print(f"\n{strategy}:")
    for config in CONFIGS:
        for pert in PERTURBATIONS:
            path = f"{OUTPUT_DIR}/QA/{strategy}/bt/{LANG}/{config}/{LANG}-{config}-{pert}.jsonl"
            if os.path.exists(path):
                count = sum(1 for line in open(path))
                print(f"  ✓ {config}/{pert}: {count} rows")
            else:
                print(f"  ✗ MISSING: {config}/{pert}")